In [22]:
import os
import math
import csv
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict, OrderedDict
from itertools import count
from typing import Dict, Counter, Any, List

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# from Common.Utils import save_training_results


In [23]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 100
    n_nodes: int = 3
    n_users: int = 200
    step_size: float = 10.0
    arrival_rate: float = 60.0  # users per minute
    zipf_alpha: float = 0.8
    n_videos: int = 500
    n_gops: int = 90
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.25  # 25% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    buffer_capacity: int = 2000
    window_len: int = 3  # LSTM sequence length (history window)
    nb_interval: int = 200  # train every 200 requests

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

In [24]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    soft_hits
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'soft_hits'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': total_reward,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'soft_hits': soft_hits
        })


In [25]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]

        self.access_order = []
        
        print(f"NetworkAdapter initialized with capacity: {self.C} videos, {self.k} tiles per video")

    def _cache_video(self, vid, bitmap: np.ndarray) -> None:
        bitmap[vid, 0, :, :] = 1

    def _evict_video(self, vid: int, bitmap: np.ndarray) -> None:
        bitmap[vid, :, :, :] = 0
    
    def _evict_tile(self, vid: int, tile: int, bitmap: np.ndarray) -> None:
        bitmap[vid, 1, tile, :] = 0

    def cache_video(self, vid: int):
        vid_idx = self.get_video_cache_idx(vid)

        if vid_idx != -1:
            self.access_order.remove(vid)
            self.access_order.append(vid)

            return

        self.access_order.append(vid)

        if len(self.access_order) > self.C:
            evict_vid = self.access_order.pop(0)
            evict_idx = self.get_video_cache_idx(evict_vid)

            vid_idx = evict_idx
        else:
            vid_idx = self.video_cache_index.index(-1)

        self.video_cache_index[vid_idx] = vid
        self.tile_cache_index[vid_idx] = [-1] * self.k

    def has_video_base_layer(self, vid: int) -> bool:
        return vid in self.video_cache_index

    def get_video_cache_idx(self, vid: int) -> int:
        for idx, v in enumerate(self.video_cache_index):
            if v == vid:
                return idx
        return -1

    def calc_cache_hits(
        self, 
        vid: int, 
        viewport: list[int], 
    ) -> tuple[int, float]:

        vid_idx = self.get_video_cache_idx(vid)
        if vid_idx == -1:
            return 0, 0.0

        hits = 12
        distortion = 30.0
        
        tiles_cached = self.tile_cache_index[vid_idx]

        for t_idx in viewport:
            if t_idx in tiles_cached:
                hits += 1
                distortion += 2.5

        return hits, distortion

    def last_sample_replication(
        self, 
        vid: int, 
        gop: int, 
        viewport: list[int],
    ):        
        vid_idx = self.get_video_cache_idx(vid)

        for i, tile in enumerate(viewport):
            self.tile_cache_index[vid_idx][i] = int(tile)

    def get_next_user_gop(self, u: int, gop: int, cfg: Config) -> list[int]:
        viewport = self.env.users_env.users_viewport_tiles[u][gop]
        return np.array(
            [y * cfg.n + x for x, y in viewport], dtype=int
        )
    
    def reset(self):
        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]
        self.access_order = []

        return self.env.reset()
    
    def env_is_done(self) -> bool:
        return self.env.users_env.users_done >= self.env.users_env.n_users

In [26]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()

    # 2. Initialize Environment
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.zipf_alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    net_adapter = NetworkAdapter(env, cfg)
    obs, info = net_adapter.reset()

    for episode in range(cfg.n_episodes):        
        cache_hits = 0
        cache_misses = 0
        soft_hits = 0.0
        total_reward = 0.0
        avg_psnr = []

        obs, info = net_adapter.reset()
        
        for step in count():

            # Get active users (not finished all GOPs)
            reqs_state = info['users_requests']
            active_users = [
                req for req in reqs_state if req['gop'] < cfg.n_gops
            ]

            # Count active users per DU
            active_users_per_du = Counter(req["p"] for req in active_users)

            # Get current cache bitmap ( Videos x Layers x Tiles x GOPs )
            bitmap = net_adapter.env.mec_cache.get_cache_bitmap()

            # Main Loop. Process each active user request
            for req in active_users:
                u, p, v, g, tiles = req['u'], req['p'], req['video'], req['gop'], req["tiles"]
                
                viewport = req['viewport'] if req['viewport'] is not None else []
                next_viewport = net_adapter.get_next_user_gop(u, g, cfg)
                
                has_base_layer = net_adapter.has_video_base_layer(v)

                if g == 0:
                    net_adapter.cache_video(vid=v)
                elif g > 0 and has_base_layer:
                    net_adapter.last_sample_replication(v, g, viewport)
                
                # Compute Cache Performance
                ch, distortion = net_adapter.calc_cache_hits(v, next_viewport)

                cache_hits += ch
                cache_misses += len(tiles) - ch
                soft_hits += float(ch) / 16.0 if len(tiles) > 0 else 0.0

                # print(
                #     f"Step {step}, User {u}, "
                #     f"Video: {v}, Gop: {g}, Viewport: {viewport}, "
                #     f"Cached Video: {net_adapter.video_cache_index[net_adapter.get_video_cache_idx(v)] if net_adapter.get_video_cache_idx(v) != -1 else 'N/A'}, "
                #     f"Cached Viewport: {net_adapter.tile_cache_index[net_adapter.get_video_cache_idx(v)] if net_adapter.get_video_cache_idx(v) != -1 else 'N/A'}, "
                #     f"Next Viewport: {next_viewport} \n"
                #     f"Cache Hits: {cache_hits}, "
                #     f"Misses: {cache_misses}, "
                #     f"Soft Hits: {soft_hits:.2f}, "
                #     f"Current Cache Hit: {ch}"
                #     # f"Hit Ratio: {hit_ratio:.2f}\n"
                # )

            # Updates in the state after processing all active users
            reqs_next_state = net_adapter.env.users_env.step(
                None, bitmap
            )

            if net_adapter.env_is_done():
                break

            info = {
                'users_requests': reqs_next_state
            }

            print(f"Step {step}, Active Users: {len(active_users)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Next Request State: {reqs_next_state}")
            print("----------------------------------------------------------------")
        
        filename = (
            f"lsr_E{cfg.n_episodes}_U{cfg.n_users}_"
            f"g{cfg.gamma}_"
            f"V{cfg.n_videos}_G{cfg.n_gops}_L{cfg.n_layers}_n{cfg.n}_m{cfg.m}_"
            f"cap{cfg.cache_size}_AR{cfg.arrival_rate}_Z{cfg.zipf_alpha}.csv"
        )

        save_training_results(
            # path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            path_=r'c:\Users\es25591\Workspace\CacheVideoPredict360\Results',
            filename=filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            soft_hits=soft_hits
        )

--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 125 videos, 4 tiles per video
Step 0, Active Users: 0
----------------------------------------------------------------
Step 1, Active Users: 0
----------------------------------------------------------------
Step 2, Active Users: 0
----------------------------------------------------------------
Step 3, Active Users: 1
----------------------------------------------------------------
Step 4, Active Users: 2
----------------------------------------------------------------
Step 5, Active Users: 3
----------------------------------------------------------------
Step 6, Active Users: 4
----------------------------------------------------------------
Step 7, Active Users: 4
----------------------------------------------------------------
Step 8, Active Users: 4
----------------------------------------------------------------
Step 9, Active Users: 5
----------------------------------------------------------------
St

KeyboardInterrupt: 

In [34]:
# Parameters
num_users = 200
lambda_rate = 10  # requests per minute
chunks_per_video = 30
chunk_time = 1  # seconds per chunk (example)

# Convert rate to requests per second
lambda_rate_per_sec = lambda_rate / 60  

# Store results
user_requests = {}

for user_id in range(num_users):
    # Total viewing time for this user
    viewing_time = chunks_per_video * chunk_time  # seconds
    
    # Number of requests during viewing time
    num_requests = np.random.poisson(lam=lambda_rate_per_sec * viewing_time)
    
    # Generate inter-arrival times (exponential distribution)
    inter_arrival_times = np.random.exponential(scale=1/lambda_rate_per_sec, size=num_requests)
    
    # Arrival timeline
    arrival_times = np.cumsum(inter_arrival_times)
    
    # Only keep arrivals that happen before video ends
    arrival_times = arrival_times[arrival_times <= viewing_time]
    
    user_requests[user_id] = arrival_times

# Example: print first 3 users
for u in range(3):
    print(f"User {u} generated {len(user_requests[u])} requests at times {user_requests[u]}")

User 0 generated 4 requests at times [ 9.94067621 17.81019565 19.09206879 21.88407463]
User 1 generated 5 requests at times [ 2.17020552  5.12621243  8.3660102  15.23145287 29.06710735]
User 2 generated 7 requests at times [ 1.38658634  3.13932507 16.31287139 19.90349812 20.12364715 20.23139716
 20.93921349]


In [45]:
# Parameters
num_users = 200
lambda_rate = 10  # requests per minute
chunks_per_video = 30
chunk_time = 1  # seconds per chunk

# Convert rate to requests per second
lambda_rate_per_sec = lambda_rate / 60  

# Store results
user_requests = {}

for user_id in range(num_users):
    requests_for_user = []
    current_time = 0
    
    for chunk in range(1, chunks_per_video + 1):
        # Viewing time for this chunk
        viewing_time = chunk_time
        
        # Expected number of requests during this chunk
        num_requests = np.random.poisson(lam=lambda_rate_per_sec * viewing_time)
        
        # Generate inter-arrival times for this chunk
        inter_arrival_times = np.random.exponential(scale=1/lambda_rate_per_sec, size=num_requests)
        arrival_times = current_time + np.cumsum(inter_arrival_times)
        
        # Only keep arrivals that happen before chunk ends
        arrival_times = arrival_times[arrival_times <= current_time + viewing_time]
        
        # Record requests with chunk info
        for t in arrival_times:
            requests_for_user.append((chunk, t))
        
        # Advance current time to next chunk
        current_time += viewing_time
    
    user_requests[user_id] = requests_for_user

# Example: print first 2 users
for u in range(2):
    print(f"User {u} generated {len(user_requests[u])} requests")
    print(user_requests[u][:10])  # show first 10 requests

User 0 generated 0 requests
[]
User 1 generated 1 requests
[(18, np.float64(17.768034661295125))]
